INFO511 NHANES wrangling-v1. Selected public CDC August 2021–August 2023 data; two synthetic learners. This purposive subset supports data engineering only, not population or medical conclusions.


SEQN, RIDAGEYR, reading, systolic_mm_hg, bpxo_match.


One source DEMO row is one selected participant; one tidy row is that participant and reading 1, 2 or 3.


In [ ]:
import pandas as pd
demographics = pd.read_csv("data/DEMO_L_selected.csv")
bp_readings = pd.read_csv("data/BPXO_L_selected.csv")
demographics_before = demographics.copy(deep=True)
bp_readings_before = bp_readings.copy(deep=True)
# Predicted: SEQN, RIDAGEYR, reading, systolic_mm_hg, bpxo_match.
# One observation is one selected participant and one reading.
demographics


Use DEMO as the left anchor; validate a one-to-one SEQN merge with BPXO. Melt exactly three systolic columns. Keep unmatched participants and every missing position; label file membership without inferring absence reasons. Preserve both source frames. Save the five-column tidy CSV and independent checks; see README.md.


In [ ]:
joined = demographics.merge(bp_readings, on="SEQN", how="left", validate="one_to_one", indicator=True)
joined["bpxo_match"] = joined["_merge"].map({"both": "matched", "left_only": "no_BPXO_record"}).astype("string")
reading_columns = ["BPXOSY1", "BPXOSY2", "BPXOSY3"]
tidy_nhanes = joined.melt(id_vars=["SEQN", "RIDAGEYR", "bpxo_match"], value_vars=reading_columns,
                   var_name="reading", value_name="systolic_mm_hg")
tidy_nhanes["reading"] = tidy_nhanes["reading"].str.removeprefix("BPXOSY").astype("int64")
tidy_nhanes = tidy_nhanes[["SEQN", "RIDAGEYR", "reading", "systolic_mm_hg", "bpxo_match"]].sort_values(["SEQN", "reading"]).reset_index(drop=True)
assert len(tidy_nhanes) == 3 * len(demographics)
assert not tidy_nhanes.duplicated(["SEQN", "reading"]).any()
assert tidy_nhanes.groupby("SEQN")["reading"].apply(set).map(lambda readings: readings == {1, 2, 3}).all()
assert tidy_nhanes["SEQN"].nunique() == len(demographics)
assert tidy_nhanes["bpxo_match"].value_counts().to_dict() == {"matched": 168, "no_BPXO_record": 24}
assert tidy_nhanes["systolic_mm_hg"].isna().sum() == 58
assert tidy_nhanes.groupby("reading")["systolic_mm_hg"].apply(lambda values: values.isna().sum()).to_dict() == {1: 16, 2: 18, 3: 24}
expected = joined.set_index("SEQN")[reading_columns].sort_index()
roundtrip = tidy_nhanes.pivot(index="SEQN", columns="reading", values="systolic_mm_hg")
roundtrip.columns = ["BPXOSY" + str(reading) for reading in roundtrip.columns]
roundtrip.columns.name = None
pd.testing.assert_frame_equal(expected, roundtrip, check_dtype=False)
for row in tidy_nhanes.itertuples(index=False):
    anchor = demographics.loc[demographics["SEQN"] == row.SEQN].iloc[0]
    assert row.RIDAGEYR == anchor.RIDAGEYR
    present = bp_readings.loc[bp_readings["SEQN"] == row.SEQN]
    assert row.bpxo_match == ("matched" if len(present) else "no_BPXO_record")
    original = present.iloc[0]["BPXOSY" + str(row.reading)] if len(present) else float("nan")
    assert (pd.isna(original) and pd.isna(row.systolic_mm_hg)) or original == row.systolic_mm_hg
pd.testing.assert_frame_equal(demographics, demographics_before)
pd.testing.assert_frame_equal(bp_readings, bp_readings_before)
tidy_nhanes.to_csv("tidy-nhanes.csv", index=False)
tidy_nhanes


Authored synthetic fixture. Execution counts and actual outputs, when present, are the execution evidence; source creation establishes neither a kernel run nor human assessment quality.
